In [56]:
import pandas as pd
import re
import string
import nltk

from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [57]:
df = pd.read_csv("IMDb Dataset.csv")

print(df.head())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [58]:
print("Dataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nTarget Column Counts:")
print(df["sentiment"].value_counts())


Dataset Shape:
(50000, 2)

Missing Values:
review       0
sentiment    0
dtype: int64

Duplicate Rows:
418

Target Column Counts:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [59]:
df.drop_duplicates(inplace=True)

print("\nShape After Removing Duplicates:")
print(df.shape)


Shape After Removing Duplicates:
(49582, 2)


In [60]:
X_train, X_test, y_train, y_test = train_test_split(
    df["review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))


Training Samples: 39665
Testing Samples: 9917


In [61]:
# Lowercase
X_train = X_train.str.lower()

# Remove URLs
X_train = X_train.str.replace(
    r'http\S+|www\S+',
    '',
    regex=True
)

# Remove Numbers
X_train = X_train.str.replace(
    r'\d+',
    '',
    regex=True
)

# Remove Punctuation
X_train = X_train.str.replace(
    f'[{string.punctuation}]',
    '',
    regex=True
)

# Remove Extra Spaces
X_train = X_train.str.replace(
    r'\s+',
    ' ',
    regex=True
).str.strip()

print("\nCleaned Training Reviews:")
print(X_train.head())


Cleaned Training Reviews:
7837     i really liked the movie the emporers new groo...
4814     i decided to watch this movie because it has b...
35458    its very hard to say just what was going on wi...
3446     this scifi adventure is not the best and by no...
24478    around the late s animator don bluth frustrate...
Name: review, dtype: object


In [62]:
# Lowercase
X_test = X_test.str.lower()

# Remove URLs
X_test = X_test.str.replace(r'http\S+|www\S+', '', regex=True)

# Remove Numbers
X_test = X_test.str.replace(r'\d+', '', regex=True)

# Remove Punctuation
X_test = X_test.str.replace(f'[{string.punctuation}]', '', regex=True)

# Remove Extra Spaces
X_test = X_test.str.replace(r'\s+', ' ', regex=True).str.strip()

# Tokenization
test_tokens = X_test.apply(word_tokenize)


In [63]:
train_tokens = X_train.apply(word_tokenize)

print("\nTokenized Reviews:")
print(train_tokens.head())



Tokenized Reviews:
7837     [i, really, liked, the, movie, the, emporers, ...
4814     [i, decided, to, watch, this, movie, because, ...
35458    [its, very, hard, to, say, just, what, was, go...
3446     [this, scifi, adventure, is, not, the, best, a...
24478    [around, the, late, s, animator, don, bluth, f...
Name: review, dtype: object


In [64]:
all_tokens = []

for tokens in train_tokens:
    all_tokens.extend(tokens)

print("\nTotal Tokens:", len(all_tokens))
print("Vocabulary Size:", len(set(all_tokens)))



Total Tokens: 9103497
Vocabulary Size: 155437


In [65]:
stemmer = PorterStemmer()

train_stemmed = train_tokens.apply(
    lambda words: [stemmer.stem(word) for word in words]
)

print("\nStemmed Reviews:")
print(train_stemmed.head())



Stemmed Reviews:
7837     [i, realli, like, the, movi, the, empor, new, ...
4814     [i, decid, to, watch, thi, movi, becaus, it, h...
35458    [it, veri, hard, to, say, just, what, wa, go, ...
3446     [thi, scifi, adventur, is, not, the, best, and...
24478    [around, the, late, s, anim, don, bluth, frust...
Name: review, dtype: object


In [66]:
stem_tokens = []

for words in train_stemmed:
    stem_tokens.extend(words)

print("\nTotal Stemmed Tokens:", len(stem_tokens))
print("Stem Vocabulary Size:", len(set(stem_tokens)))


Total Stemmed Tokens: 9103497
Stem Vocabulary Size: 120649


In [67]:
lemmatizer = WordNetLemmatizer()

train_lemmatized = train_tokens.apply(
    lambda words: [lemmatizer.lemmatize(word) for word in words]
)

print("\nLemmatized Reviews:")
print(train_lemmatized.head())



Lemmatized Reviews:
7837     [i, really, liked, the, movie, the, emporers, ...
4814     [i, decided, to, watch, this, movie, because, ...
35458    [it, very, hard, to, say, just, what, wa, goin...
3446     [this, scifi, adventure, is, not, the, best, a...
24478    [around, the, late, s, animator, don, bluth, f...
Name: review, dtype: object


In [68]:
lemma_tokens = []

for words in train_lemmatized:
    lemma_tokens.extend(words)

print("\nTotal Lemmatized Tokens:", len(lemma_tokens))
print("Lemma Vocabulary Size:", len(set(lemma_tokens)))


Total Lemmatized Tokens: 9103497
Lemma Vocabulary Size: 145086


In [69]:
comparison = pd.DataFrame({
    "Original Review": df.loc[X_train.index, "review"],
    "Cleaned Review": X_train,
    "Tokenized": train_tokens,
    "Stemmed": train_stemmed,
    "Lemmatized": train_lemmatized
})

comparison.head()

,Original Review,Cleaned Review,Tokenized,Stemmed,Lemmatized
7837,I really liked the movie 'The Emporer's New Gr...,i really liked the movie the emporers new groo...,"[i, really, liked, the, movie, the, emporers, ...","[i, realli, like, the, movi, the, empor, new, ...","[i, really, liked, the, movie, the, emporers, ..."
4814,I decided to watch this movie because it has b...,i decided to watch this movie because it has b...,"[i, decided, to, watch, this, movie, because, ...","[i, decid, to, watch, thi, movi, becaus, it, h...","[i, decided, to, watch, this, movie, because, ..."
35458,It's very hard to say just what was going on w...,its very hard to say just what was going on wi...,"[its, very, hard, to, say, just, what, was, go...","[it, veri, hard, to, say, just, what, wa, go, ...","[it, very, hard, to, say, just, what, wa, goin..."
3446,This sci-fi adventure is not the best and by n...,this scifi adventure is not the best and by no...,"[this, scifi, adventure, is, not, the, best, a...","[thi, scifi, adventur, is, not, the, best, and...","[this, scifi, adventure, is, not, the, best, a..."
24478,"Around the late 1970's, animator Don Bluth, fr...",around the late s animator don bluth frustrate...,"[around, the, late, s, animator, don, bluth, f...","[around, the, late, s, anim, don, bluth, frust...","[around, the, late, s, animator, don, bluth, f..."


In [70]:
results = pd.DataFrame({
    "Stage": [
        "Tokenization",
        "Stemming",
        "Lemmatization"
    ],
    "Total Tokens": [
        len(all_tokens),
        len(stem_tokens),
        len(lemma_tokens)
    ],
    "Vocabulary Size": [
        len(set(all_tokens)),
        len(set(stem_tokens)),
        len(set(lemma_tokens))
    ]
})

print(results)

           Stage  Total Tokens  Vocabulary Size
0   Tokenization       9103497           155437
1       Stemming       9103497           120649
2  Lemmatization       9103497           145086


In [71]:
from sklearn.feature_extraction.text import CountVectorizer

# One-Hot Encoding
onehot_vectorizer = CountVectorizer(binary=True)

X_train_onehot = onehot_vectorizer.fit_transform(X_train)
X_test_onehot = onehot_vectorizer.transform(X_test)

print("Training Shape:", X_train_onehot.shape)
print("Testing Shape:", X_test_onehot.shape)

Training Shape: (39665, 154667)
Testing Shape: (9917, 154667)


In [72]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_onehot, y_train)

y_pred_onehot = model.predict(X_test_onehot)

In [73]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

accuracy_onehot = accuracy_score(y_test, y_pred_onehot)
print("Accuracy:", accuracy_onehot)
print("One-Hot Classification Report:")
print(classification_report(y_test, y_pred_onehot))

Accuracy: 0.8828274679842695
One-Hot Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.88      0.88      4939
    positive       0.88      0.89      0.88      4978

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



In [74]:
from sklearn.feature_extraction.text import CountVectorizer
# Bag of Words
bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print("Training Shape:", X_train_bow.shape)
print("Testing Shape:", X_test_bow.shape)

Training Shape: (39665, 154667)
Testing Shape: (9917, 154667)


In [75]:
model_bow = LogisticRegression(max_iter=1000)

model_bow.fit(X_train_bow, y_train)

y_pred_bow = model_bow.predict(X_test_bow)

In [76]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_bow = accuracy_score(y_test, y_pred_bow)

print("Bag of Words Accuracy:", accuracy_bow)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_bow))

Bag of Words Accuracy: 0.8879701522637895

Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.87      0.89      4939
    positive       0.88      0.90      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("Training Shape:", X_train_tfidf.shape)
print("Testing Shape:", X_test_tfidf.shape)

Training Shape: (39665, 154667)
Testing Shape: (9917, 154667)


In [81]:
from sklearn.linear_model import LogisticRegression

model_tfidf = LogisticRegression(max_iter=1000)

model_tfidf.fit(X_train_tfidf, y_train)

y_pred_tfidf = model_tfidf.predict(X_test_tfidf)


In [82]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("TF-IDF Accuracy:", accuracy_tfidf)

print("\nTF-IDF Classification Report:")
print(classification_report(y_test, y_pred_tfidf))

TF-IDF Accuracy: 0.8922053040233942

TF-IDF Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4939
    positive       0.88      0.91      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [83]:
from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [84]:
print("Training Word2Vec...")

w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

Training Word2Vec...


In [85]:
keras_tokenizer = Tokenizer()

keras_tokenizer.fit_on_texts(X_train)

vocab_size = len(keras_tokenizer.word_index) + 1
max_len = 100

In [86]:
X_train_seq = keras_tokenizer.texts_to_sequences(X_train)
X_test_seq = keras_tokenizer.texts_to_sequences(X_test)

In [87]:
X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_len,
    padding="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_len,
    padding="post"
)

In [88]:
embedding_matrix = np.zeros((vocab_size, 100))

for word, i in keras_tokenizer.word_index.items():

    if word in w2v_model.wv:

        embedding_matrix[i] = w2v_model.wv[word]

In [90]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from sklearn.metrics import classification_report

In [91]:
y_train_nn = y_train.map({
    "negative": 0,
    "positive": 1
})

y_test_nn = y_test.map({
    "negative": 0,
    "positive": 1
})

In [92]:
model_w2v = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        weights=[embedding_matrix],
        trainable=True,
        input_length=max_len
    ),

    GlobalAveragePooling1D(),

    Dense(64, activation="relu"),

    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [93]:
model_w2v.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
print("Training Word2Vec Model...")

history = model_w2v.fit(
    X_train_pad,
    y_train_nn,
    epochs=2,
    batch_size=16,
    validation_split=0.2
)

Training Word2Vec Model...
Epoch 1/2
992/992 ━━━━━━━━━━━━━━━━━━━━ 112s 113ms/step - accuracy: 0.8813 - loss: 0.2813 - val_accuracy: 0.8568 - val_loss: 0.3345
Epoch 2/2
992/992 ━━━━━━━━━━━━━━━━━━━━ 107s 107ms/step - accuracy: 0.9130 - loss: 0.2207 - val_accuracy: 0.8542 - val_loss: 0.3509


In [96]:
loss, accuracy = model_w2v.evaluate(
    X_test_pad,
    y_test_nn
)

print("Test Accuracy:", accuracy)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8552 - loss: 0.3483
Test Accuracy: 0.8551981449127197


In [97]:
y_pred = model_w2v.predict(X_test_pad)
y_pred = (y_pred > 0.5).astype(int)
print("\nWord2Vec Classification Report")

print(classification_report(
    y_test_nn,
    y_pred
))

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Word2Vec Classification Report
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      4939
           1       0.87      0.83      0.85      4978

    accuracy                           0.86      9917
   macro avg       0.86      0.86      0.86      9917
weighted avg       0.86      0.86      0.86      9917

